In [ ]:
import sys

path_to_add = "../gloabl_files/ml_binary_classification_gridsearch_hyperOpt/"
sys.path.append(path_to_add)


In [ ]:
import logging
import os
from pathlib import Path
import yaml


import pandas as pd
from IPython.display import display

from ml_grid.util.impute_data_for_pipe import (
    mean_impute_dataframe,
    save_missing_percentage,
)

from ml_grid.util.synthetic_data_generator import generate_synthetic_ts_data

# --- 1. Setup Logging ---
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# --- 2. Generate Synthetic Longitudinal Data and the Ground-Truth Map ---
logging.info("Generating a sample synthetic longitudinal dataset using the importable function...")
ts_df, important_feature_map = generate_synthetic_ts_data(
    n_instances=50,
    n_timepoints=10,
    n_features=15,
    n_outcome_vars=1,
    feature_strength=0.7,
    percent_important_features=0.2,
    percent_missing=0.1,
    start_date="2022-01-01",
    verbose=True,
)

print("\n--- Generated Longitudinal DataFrame Info (Before Imputation) ---")
print(f"Shape: {ts_df.shape}")
print(f"Unique patients: {ts_df['client_idcode'].nunique()}")
print(f"Timepoints per patient: {ts_df.groupby('client_idcode').size().unique().tolist()}")
print(f"Total NaNs present: {ts_df.isnull().sum().sum()}")
print("Sample of data with missing values:")
display(ts_df.head(10))
print("-----------------------------------------------------------------")


# --- 3. Calculate and Save the Percentage of Missing Values ---
# Only calculate over feature columns — exclude structural and outcome columns.
outcome_columns = list(important_feature_map.keys())
non_feature_cols = ["client_idcode", "timestamp"] + outcome_columns
feature_cols = [c for c in ts_df.columns if c not in non_feature_cols]

missing_pickle_filename = "percent_missing_synthetic_ts_data_generated.pkl"
print(f"\nCalculating missing value percentages and saving to '{missing_pickle_filename}'...")
save_missing_percentage(ts_df[feature_cols], output_file=missing_pickle_filename)
print("✅ Missing value pickle file saved.")


# --- 4. Perform Mean Imputation ---
print("\nPerforming mean imputation on the dataset...")
# Exclude structural and outcome columns from imputation
imputed_ts_df = mean_impute_dataframe(data=ts_df.copy(), y_vars=non_feature_cols)
print(f"Imputation complete. NaNs present after imputation: {imputed_ts_df.isnull().sum().sum()}")
print("✅ Mean imputation successful.")


# --- 5. Save the Imputed Data to the Final CSV File ---
output_csv_filename = "synthetic_ts_data_generated.csv"
imputed_ts_df.to_csv(output_csv_filename, index=False)
print(f"\nImputed data saved to '{output_csv_filename}'")
print("✅ Final CSV file saved.")

print("\n--- Final Imputed Longitudinal DataFrame ---")
display(imputed_ts_df.head(10))
print("--------------------------------------------")

In [ ]:
pwd

In [ ]:
import sys

In [ ]:
#os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="1"

In [ ]:
pwd

In [ ]:
import ipywidgets as ipw
output = ipw.Output()

In [ ]:
import warnings
warnings.filterwarnings('ignore') 

In [ ]:
import os
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"

In [ ]:
#%pip install hyperopt

In [ ]:
from hyperopt import fmin, tpe, hp

In [ ]:
import logging

# --- 1. Setup Logging ---
# This allows you to see the informative output from the generator and other steps.
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

logger = logging.getLogger('matplotlib.font_manager')

# Set the logging level to suppress debug messages
logger.setLevel(logging.INFO)

In [ ]:
grid = {
            
            'resample' : ['undersample', 'oversample', None],
            'scale'    : [True, False],
            'feature_n': [100, 95, 75, 50, 25, 5],
            'param_space_size':['medium', 'xsmall'],
            'n_unique_out': [10],
            'outcome_var_n':['1'],
                            'percent_missing':[99, 95, 80],  #n/100 ex 95 for 95% # 99.99, 99.5, 9
                            'corr':[0.98, 0.85, 0.5, 0.25],
                            'data':[{'age':[True, False],
                                    'sex':[True, False],
                                    'bmi':[True],
                                    'ethnicity':[True, False],
                                    'bloods':[True, False],
                                    'diagnostic_order':[True, False],
                                    'drug_order':[True, False],
                                    'annotation_n':[True, False],
                                    'meta_sp_annotation_n':[True, False],
                                    'annotation_mrc_n':[True, False],
                                    'meta_sp_annotation_mrc_n':[True, False],
                                    'core_02':[False],
                                    'bed':[False],
                                    'vte_status':[True],
                                    'hosp_site':[True],
                                    'core_resus':[False],
                                    'news':[False],
                                    'date_time_stamp':[ False]
                                    
                                    }]
        }

In [ ]:

space = {
    'resample': hp.choice('resample', ['undersample', 'oversample', None]),
    'scale': hp.choice('scale', [True, False]),
    'feature_n': hp.choice('feature_n', [100, 95, 75, 50, 25, 5]),
    'param_space_size': hp.choice('param_space_size', ['medium', 'xsmall']),
    'n_unique_out': hp.choice('n_unique_out', [10]),
    'outcome_var_n': hp.choice('outcome_var_n', ['1']),
    'percent_missing': hp.choice('percent_missing', [99, 95, 80]),
    'corr': hp.choice('corr', [0.98, 0.85, 0.5, 0.25]),
    'data': {
        'age': hp.choice('age', [True, False]),
        'sex': hp.choice('sex', [True, False]),
        'bmi': hp.choice('bmi', [True]),
        'ethnicity': hp.choice('ethnicity', [True, False]),
        'bloods': hp.choice('bloods', [True, False]),
        'diagnostic_order': hp.choice('diagnostic_order', [True, False]),
        'drug_order': hp.choice('drug_order', [True, False]),
        'annotation_n': hp.choice('annotation_n', [True, False]),
        'meta_sp_annotation_n': hp.choice('meta_sp_annotation_n', [True, False]),
        'annotation_mrc_n': hp.choice('annotation_mrc_n', [True, False]),
        'meta_sp_annotation_mrc_n': hp.choice('meta_sp_annotation_mrc_n', [True, False]),
        'core_02': hp.choice('core_02', [False]),
        'bed': hp.choice('bed', [False]),
        'vte_status': hp.choice('vte_status', [True]),
        'hosp_site': hp.choice('hosp_site', [True]),
        'core_resus': hp.choice('core_resus', [False]),
        'news': hp.choice('news', [False]),
        'date_time_stamp': hp.choice('date_time_stamp', [False])
    }
}


In [ ]:
# Breast cancer sample space:

space_breast_cancer = {
    'resample': hp.choice('resample', ['undersample', 'oversample', None]),
    'scale': hp.choice('scale', [True, False]),
    'feature_n': hp.choice('feature_n', [ 25, 5]),
    'param_space_size': hp.choice('param_space_size', ['medium', 'xsmall']),
    'n_unique_out': hp.choice('n_unique_out', [10]),
    'outcome_var_n': hp.choice('outcome_var_n', ['1']),
    'percent_missing': hp.choice('percent_missing', [99, 95, 80]),
    'corr': hp.choice('corr', [0.98, 0.85, 0.5, 0.25]),
    'data': {
        'age': hp.choice('age', [False]),
        'sex': hp.choice('sex', [ False]),
        'bmi': hp.choice('bmi', [False]),
        'ethnicity': hp.choice('ethnicity', [ False]),
        'bloods': hp.choice('bloods', [True, ]),
        'diagnostic_order': hp.choice('diagnostic_order', [ False]),
        'drug_order': hp.choice('drug_order', [ False]),
        'annotation_n': hp.choice('annotation_n', [ False]),
        'meta_sp_annotation_n': hp.choice('meta_sp_annotation_n', [ False]),
        'annotation_mrc_n': hp.choice('annotation_mrc_n', [ False]),
        'meta_sp_annotation_mrc_n': hp.choice('meta_sp_annotation_mrc_n', [ False]),
        'core_02': hp.choice('core_02', [False]),
        'bed': hp.choice('bed', [False]),
        'vte_status': hp.choice('vte_status', [False]),
        'hosp_site': hp.choice('hosp_site', [False]),
        'core_resus': hp.choice('core_resus', [False]),
        'news': hp.choice('news', [False]),
        'date_time_stamp': hp.choice('date_time_stamp', [False]),
    }
}

In [ ]:

# --- Load Configuration ---
# Load the models and search space from your YAML configuration file.

config_path = Path('/home/samorah/gloabl_files/ml_binary_classification_gridsearch_hyperOpt/config_hyperopt.yml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# This creates the 'model_class_dict' variable that your objective function needs.
model_class_dict = config.get('ts_models', {})

if not model_class_dict:
    raise ValueError("Could not find or load 'ts_models' from the configuration file.")

print(f"Successfully loaded {len(model_class_dict)} time-series models for the Hyperopt search.")



In [ ]:
import datetime
import random
import logging
from pathlib import Path

from IPython.display import clear_output
from hyperopt import STATUS_FAIL

from ml_grid.pipeline.data import NoFeaturesError, pipe
from ml_grid.pipeline import main
from ml_grid.util.create_experiment_directory import create_experiment_directory
from ml_grid.util.project_score_save import project_score_save_class

random.seed(1234)

base_project_dir = "HFE_ML_experiments_ts"
experiment_name = f"hyperopt_run_{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M')}"


# --- Logging Setup ---
logger = logging.getLogger(__name__)

class ByteflowFilter(logging.Filter):
    def filter(self, record):
        return record.name.startswith('numba.core.byteflow')

logger.addFilter(ByteflowFilter())

# --- Setup Experiment Directory and Logging ---
# This is the main directory for the entire Hyperopt search.
experiment_dir = create_experiment_directory(
    base_dir=base_project_dir,       # From cell 7
    additional_naming=experiment_name # From cell 7
)
experiment_dir = Path(experiment_dir)
print(f"Main experiment directory: {experiment_dir.resolve()}")

# Initialize the project-level score log within the main experiment directory
project_score_save_class(experiment_dir)

input_csv_path = Path('synthetic_ts_data_generated.csv')

def objective(local_param_dict, outcome_var=None):
    """The objective function that Hyperopt will minimize (time series mode)."""
    clear_output(wait=True)
    print(f"Evaluating for outcome: {outcome_var}")
    print(f"Params: {local_param_dict}")

    # A unique ID for this specific trial
    trial_idx = random.randint(0, 9999999999)

    try:
        ml_grid_object = pipe(
            file_name=str(input_csv_path.resolve()),
            drop_term_list=['chrom', 'hfe', 'phlebo'],
            local_param_dict=local_param_dict,
            base_project_dir=base_project_dir,
            experiment_dir=experiment_dir,
            additional_naming=experiment_name,
            test_sample_n=0,
            param_space_index=trial_idx,
            model_class_dict=model_class_dict,
            outcome_var_override=outcome_var,
            time_series_mode=True          # <-- retained from old TS workflow
        )

        # Run the modeling pipeline and get the best score for this trial
        errors, highest_score = main.run(
            local_param_dict=local_param_dict, ml_grid_object=ml_grid_object
        ).execute()

        return {
            'loss': 1 - float(highest_score),  # Hyperopt minimizes, so we use 1 - AUC
            'status': 'ok'
        }
    except NoFeaturesError as e:
        print(f"Skipping trial due to NoFeaturesError: {e}")
        return {'status': STATUS_FAIL, 'loss': float('inf')}
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()
        raise e


In [ ]:
#objective(next(grid_iter_obj))

In [ ]:
print("done")

In [ ]:
# results_df = pd.read_csv(base_project_dir + 'final_grid_score_log.csv')
    
# highest_metric_from_run = results_df[results_df['i'] == str(900424809465212743016)].sort_values(by='auc')['auc'].iloc[-1]

# highest_metric_from_run

In [ ]:
from hyperopt import hp, Trials

In [ ]:
trials = Trials()

In [ ]:
best = fmin(fn=objective,
            space=space,
            algo=tpe.suggest,
            max_evals=2,
            trials = trials,
           verbose=1
           )

In [ ]:
best

In [ ]:
import os
from datetime import datetime


# Define the parent directory from config
parent_dir = experiment_dir # Use the path from the run

# Check if the CSV is directly in the parent directory first
csv_path = os.path.join(parent_dir, 'final_grid_score_log.csv')

if not os.path.exists(csv_path):
    # If not found directly, look for it in timestamped subfolders
    # List all folders in the parent directory that match the date pattern
    folders = [f for f in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, f))]

    # Parse folder names as dates and find the latest one
    def parse_date(folder_name: str):
        """
        Parses the timestamp from the beginning of a folder name.
        Expected format: 'YYYY-MM-DD_HH-MM-SS_...'.
        """
        try:
            # The timestamp is always the first 19 characters.
            timestamp_part = folder_name[:19]
            return datetime.strptime(timestamp_part, '%Y-%m-%d_%H-%M-%S')
        except (ValueError, IndexError):
            # Return None if the folder name doesn't match the expected format or is too short.
            return None

    # Filter and sort folders by date
    folders_with_dates = [(f, parse_date(f)) for f in folders]
    folders_with_dates = [f for f in folders_with_dates if f[1] is not None]

    if folders_with_dates:
        latest_folder = max(folders_with_dates, key=lambda x: x[1])[0]  # Get the folder with the latest date
        print("latest_folder", latest_folder)

        # Construct the path to the CSV file in the latest folder
        csv_path = os.path.join(parent_dir, latest_folder, 'final_grid_score_log.csv')
    else:
        raise FileNotFoundError("No timestamped folders found and CSV not in parent directory")
else:
    print("CSV found directly in parent directory")

# Load the CSV file
df = pd.read_csv(csv_path)

# Sort the DataFrame by 'auc' column in descending order
df = df.sort_values(by='auc', ascending=False)

print(f"Total rows: {len(df)}")

# Group by outcome_variable and display the first row of each group with the highest auc
df_best = df.groupby('outcome_variable').apply(lambda x: x.iloc[0])

# Display the result
df_best.head()

In [ ]:
df_best['algorithm_implementation'].value_counts()

In [ ]:
from datetime import datetime
from pathlib import Path

import pandas as pd

# --- Configuration ---
experiments_base_dir = Path(config['experiment']['experiments_base_dir'])

# --- Find the CSV file (try multiple locations) ---

def find_csv_file():
    """
    Search for final_grid_score_log.csv in multiple locations:
    1. Project root (parent of experiments_base_dir)
    2. Directly in experiments_base_dir
    3. In the latest timestamped subfolder
    """
    # Location 1: Project root (where the notebook is run from)
    project_root = experiments_base_dir.parent
    csv_path = project_root / 'final_grid_score_log.csv'
    if csv_path.exists():
        print(f"✓ CSV found in project root: {csv_path.resolve()}")
        return csv_path

    # Location 2: Directly in experiments directory
    csv_path = project_root / experiments_base_dir / 'final_grid_score_log.csv'
    if csv_path.exists():
        print(f"✓ CSV found in experiments directory: {csv_path.resolve()}")
        return csv_path

    # Location 3: In latest timestamped subfolder
    latest_folder = find_latest_experiment_folder()
    if latest_folder:
        csv_path = latest_folder / 'final_grid_score_log.csv' # latest_folder is already absolute
        if csv_path.exists():
            print(f"✓ CSV found in latest experiment folder: {csv_path.resolve()}")
            return csv_path

    return None


def parse_date(folder_name: str):
    """
    Parses the timestamp from the beginning of a folder name.
    Expected format: 'YYYY-MM-DD_HH-MM-SS_...'.
    """
    try:
        timestamp_part = folder_name[:19]
        return datetime.strptime(timestamp_part, '%Y-%m-%d_%H-%M-%S')
    except (ValueError, IndexError):
        return None


def find_latest_experiment_folder():
    """Find the most recent timestamped experiment folder."""
    if not experiments_base_dir.exists() or not experiments_base_dir.is_dir():
        print(f"⚠ Experiments directory not found: {(project_root / experiments_base_dir).resolve()}")
        return None

    subfolders = [f for f in experiments_base_dir.iterdir() if f.is_dir()]
    folders_with_dates = [(f, parse_date(f.name)) for f in subfolders]
    valid_folders = [f for f in folders_with_dates if f[1] is not None]

    if valid_folders:
        latest_folder = max(valid_folders, key=lambda x: x[1])[0]
        print(f"Latest experiment folder: {latest_folder.name}")
        return latest_folder
    else:
        print("⚠ No valid timestamped experiment folders found.")
        return None


# --- Main Execution ---

# Find the CSV file
log_file_path = find_csv_file()

if log_file_path and log_file_path.exists():
    # Load the CSV file
    df = pd.read_csv(log_file_path)

    # Sort by AUC in descending order
    df_sorted = df.sort_values(by='auc', ascending=False)

    print(f"\n✓ Successfully loaded {len(df_sorted)} records from the log file.")

    # Group by outcome_variable and get the best result for each
    top_results_by_outcome = df_sorted.groupby('outcome_variable').first().reset_index()

    print(f"\nTop results by outcome variable ({len(top_results_by_outcome)} outcomes):\n")

    # Display the result
    display(top_results_by_outcome)

else:
    print("\n✗ Error: Could not find 'final_grid_score_log.csv' in any expected location:")
    print(f"  - {(experiments_base_dir.parent / 'final_grid_score_log.csv').resolve()}")
    print(f"  - {(project_root / experiments_base_dir / 'final_grid_score_log.csv').resolve()}")
    print(f"  - In any timestamped subfolder within {experiments_base_dir.resolve()}")

In [ ]:
import numpy as np
import pandas as pd

# Load the data
data_path = 'synthetic_ts_data_generated.csv'
data = pd.read_csv(data_path)

# Display basic information about the dataset
print("=== Dataset Information ===")
print(f"Shape of the dataset: {data.shape}")
print(f"Columns: {data.columns.tolist()}")
print("\nFirst 5 rows of the dataset:")
print(data.head())

# Check for missing values
print("\n=== Missing Values ===")
missing_values = data.isnull().sum()
print(missing_values[missing_values > 0])

# Check for constant features
print("\n=== Constant Features ===")
constant_features = [col for col in data.columns if data[col].nunique() == 1]
print(f"Constant features: {constant_features}")

# Check for features with very low variance (almost constant)
print("\n=== Low Variance Features ===")
low_variance_features = []
for col in data.columns:
    if data[col].dtype in [np.float64, np.int64]:  # Check only numeric features
        if data[col].std() < 0.01:  # Threshold for low variance
            low_variance_features.append(col)
print(f"Low variance features: {low_variance_features}")

# Check for duplicate rows
print("\n=== Duplicate Rows ===")
duplicate_rows = data.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

# Check for class distribution (if it's a classification problem)
if 'target' in data.columns:  # Replace 'target' with your actual target column name
    print("\n=== Class Distribution ===")
    print(data['target'].value_counts())

# Check for categorical features with high cardinality
print("\n=== High Cardinality Categorical Features ===")
categorical_features = data.select_dtypes(include=['object', 'category']).columns
high_cardinality_features = [col for col in categorical_features if data[col].nunique() > 100]
print(f"High cardinality categorical features: {high_cardinality_features}")

# Check for outliers in numeric features (using IQR)
print("\n=== Outliers in Numeric Features ===")
numeric_features = data.select_dtypes(include=[np.float64, np.int64]).columns
for col in numeric_features:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower_bound) | (data[col] > upper_bound)]
    if not outliers.empty:
        print(f"Outliers in {col}: {len(outliers)} rows")

# Summary of issues
print("\n=== Summary of Issues ===")
if missing_values.any():
    print(f"- Missing values found in {missing_values[missing_values > 0].index.tolist()}")
if constant_features:
    print(f"- Constant features found: {constant_features}")
if low_variance_features:
    print(f"- Low variance features found: {low_variance_features}")
if duplicate_rows > 0:
    print(f"- Duplicate rows found: {duplicate_rows}")
if high_cardinality_features:
    print(f"- High cardinality categorical features found: {high_cardinality_features}")
if not missing_values.any() and not constant_features and not low_variance_features and not duplicate_rows and not high_cardinality_features:
    print("- No major issues found in the dataset.")

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Assuming your DataFrame is named 'df'

# Get the top result for each outcome_variable by AUC
top_auc_per_outcome = df.loc[df.groupby('outcome_variable')['auc'].idxmax()]

# Sort by AUC for better visualization
top_auc_per_outcome = top_auc_per_outcome.sort_values(by='auc', ascending=False)

# Set plot style
sns.set_style("whitegrid")
plt.figure(figsize=(12, 8))

# Create barplot to show the top AUC for each outcome_variable
sns.barplot(
    x='auc',
    y='outcome_variable',
    data=top_auc_per_outcome,
    hue='nb_size',
    dodge=False,
    palette='viridis'
)

# Add titles and labels
plt.title('Top AUC for Each Outcome Variable')
plt.xlabel('AUC')
plt.ylabel('Outcome Variable')
plt.legend(title='num features')

# Display the plot
plt.show()


In [ ]:
print("done")

In [ ]:
# Import the necessary classes
import pandas as pd

from ml_grid.results_processing.core import ResultsAggregator
from ml_grid.results_processing.plot_master import MasterPlotter

# 1. Load your data using the ResultsAggregator
#    Replace with the actual path to your results and feature names file.
#    The feature_names_csv is optional but required for feature-related plots.
try:
    aggregator = ResultsAggregator(
        root_folder=config['experiment']['experiments_base_dir'],
        feature_names_csv=config['data']['file_path'])
    results_df = aggregator.aggregate_all_runs()

    # 2. Instantiate the MasterPlotter with your data
    master_plotter = MasterPlotter(results_df)

    # 3. Call the plot_all() method to generate all visualizations
    #    You can customize the primary metric and other options.
    master_plotter.plot_all(metric='auc', stratify_by_outcome=True)

except (ValueError, FileNotFoundError) as e:
    print(f"An error occurred: {e}")
    print("Please ensure your results folder path is correct and contains valid run data.")



In [ ]:
display(config)

In [ ]:
df['failed'].value_counts()

In [ ]:
df.sort_values(by='run_time', ascending=False)

In [ ]:
#old 

In [ ]:
results_df = pd.read_csv(os.path.join(experiment_dir , 'final_grid_score_log.csv'))

In [ ]:
results_df.sort_values('auc', ascending=False).iloc[0]

In [ ]:
results_df.sort_values('auc', ascending=False)

In [ ]:
# Import the necessary classes
from ml_grid.results_processing.core import ResultsAggregator
from ml_grid.results_processing.plot_master import MasterPlotter
import pandas as pd

# 1. Load your data using the ResultsAggregator
#    Replace with the actual path to your results and feature names file.
#    The feature_names_csv is optional but required for feature-related plots.
try:
    aggregator = ResultsAggregator(
        root_folder='HFE_ML_experiments_ts',
        feature_names_csv='synthetic_ts_data_generated.csv')
    results_df = aggregator.aggregate_all_runs()

    # 2. Instantiate the MasterPlotter with your data
    master_plotter = MasterPlotter(results_df)

    # 3. Call the plot_all() method to generate all visualizations
    #    You can customize the primary metric and other options.
    master_plotter.plot_all(metric='auc', stratify_by_outcome=True)

except (ValueError, FileNotFoundError) as e:
    print(f"An error occurred: {e}")
    print("Please ensure your results folder path is correct and contains valid run data.")

